# Clase 108 — Inicialización (Glorot, He)

Inicializar los pesos para que la **varianza de las activaciones y de los gradientes** se mantenga estable en el forward y el backward. **Glorot (Xavier)** para sigmoid/tanh; **He (Kaiming)** para ReLU y variantes; **LeCun** para SELU.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

## 1. Glorot uniform: límites teóricos vs empíricos

`Dense` usa `glorot_uniform` por defecto: `U(-√(6/(fan_in+fan_out)), +√(6/(fan_in+fan_out)))`.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

capa = layers.Dense(256, kernel_initializer="glorot_uniform")
capa.build((None, 784))                          # materializa los pesos
W = capa.kernel.numpy()
fan_in, fan_out = 784, 256
limite = np.sqrt(6 / (fan_in + fan_out))
print("límite teórico Glorot uniform: ±", round(limite, 4))
print("min/max empírico:", round(W.min(), 4), round(W.max(), 4))
print("var empírica:", round(W.var(), 6),
      "| var teórica 2/(fan_in+fan_out):", round(2 / (fan_in + fan_out), 6))

## 2. He normal: `Var(W) = 2/fan_in`

He compensa que ReLU pone a 0 la mitad de las salidas. Recomendado para ReLU, Leaky ReLU y ELU.

In [ ]:
capa_he = layers.Dense(256, kernel_initializer="he_normal")
capa_he.build((None, 784))
Whe = capa_he.kernel.numpy()
print("var He empírica:", round(Whe.var(), 6), "| teórica 2/fan_in:", round(2 / 784, 6))
print("bias inicial (siempre zeros por defecto):", np.unique(capa_he.bias.numpy()))

## 3. Comparar inits entrenando un MLP con ReLU

Glorot, He y un `RandomNormal(stddev=0.01)` (mala costumbre de tutoriales viejos → vanishing inmediato).

In [ ]:
def mlp(init):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(512, activation="relu", kernel_initializer=init),
        layers.Dense(256, activation="relu", kernel_initializer=init),
        layers.Dense(128, activation="relu", kernel_initializer=init),
        layers.Dense(64,  activation="relu", kernel_initializer=init),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for nombre, init in [("Glorot", "glorot_uniform"),
                     ("He normal", "he_normal"),
                     ("RandomNormal 0.01", keras.initializers.RandomNormal(stddev=0.01))]:
    print(f"{nombre:18s} params: {mlp(init).count_params()}")
    # mlp(init).fit(X_tr, y_tr, epochs=20, validation_split=0.1)

## 4. Histograma de activaciones por capa

Con He + ReLU la desviación estándar de las activaciones se conserva a lo largo de las capas (no se apaga ni explota).

In [ ]:
entrada = keras.Input(shape=(784,))
x = entrada
salidas = []
for _ in range(5):
    x = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(x)
    salidas.append(x)
sonda = keras.Model(entrada, salidas)

batch = np.random.default_rng(0).normal(size=(512, 784)).astype("float32")
for i, a in enumerate(sonda(batch), start=1):
    print(f"capa {i}: std activaciones = {float(a.numpy().std()):.4f}")

## 5. LeCun normal + SELU (self-normalizing)

SELU se auto-normaliza **solo** si se combina con `lecun_normal`. Fue la propuesta de Klambauer et al. (2017).

In [ ]:
selu_net = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(256, activation="selu", kernel_initializer="lecun_normal"),
    layers.Dense(256, activation="selu", kernel_initializer="lecun_normal"),
    layers.Dense(10,  activation="softmax"),
])
capa0 = selu_net.layers[0]
capa0.build((None, 784))
print("LeCun normal: var teórica 1/fan_in =", round(1 / 784, 6))
print("SELU requiere lecun_normal para conservar media 0 y varianza 1 por capa.")

## Ejercicios

1. **Inspección de defaults**: para `Dense(128)` sobre entrada 784, calculá la varianza empírica del kernel y compará con la teórica de Glorot.
2. **Comparación**: entrená el MLP `[512,256,128,64,10]` con Glorot, He normal y `RandomNormal(0.01)`. Graficá `val_loss`.
3. **He + tanh (combinación incorrecta)**: comparala contra Glorot + tanh y verificá que la elección importa.
4. **Reproducibilidad**: con `keras.utils.set_random_seed(42)` entrená dos veces y verificá que da idéntico.

## Conclusiones

- La idea central: `Var(W) ≈ 1/fan_in` (o promedio con `fan_out`) para preservar varianza a través de las capas.
- **Glorot** (`glorot_uniform`, default de Keras) para sigmoid/tanh; **He** (`he_normal`) para ReLU y variantes; **LeCun** (`lecun_normal`) para SELU.
- El **bias arranca en `zeros`**; inicializarlo al azar empeora el arranque.
- BatchNorm reduce la sensibilidad al init, pero un mal init aún desestabiliza las primeras épocas.
- En Transformers se usa `truncated_normal(stddev=0.02)` estilo GPT/BERT.

## ✅ Soluciones de los ejercicios

Inicialización Glorot vs He: inspección de la varianza teórica, comparación empírica, histogramas de activaciones, la combinación correcta init↔activación y reproducibilidad con semilla. Sin TF se validan por AST.

**Ej. 1 — Inspección de defaults.** La varianza empírica del kernel de una `Dense` coincide con la teórica de Glorot: `2/(fan_in+fan_out)`.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([keras.Input((784,)), layers.Dense(128)])
W = model.layers[0].kernel.numpy()
var_emp = W.var()
var_glorot = 2.0 / (784 + 128)         # Var teorica de Glorot
print(f"varianza empirica = {var_emp:.5f}  |  teorica Glorot = {var_glorot:.5f}")
assert abs(var_emp - var_glorot) < 5e-4
print("Keras usa glorot_uniform por defecto en Dense.")

**Ej. 2 — Comparación de inits.** MLP `[512,256,128,64,10]` con ReLU: Glorot vs He vs `RandomNormal(0.01)`.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(init):
    m = keras.Sequential([keras.Input((784,))] +
        [layers.Dense(u, activation="relu", kernel_initializer=init) for u in (512, 256, 128, 64)] +
        [layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for init, name in [("glorot_uniform", "Glorot"), ("he_normal", "He"),
                   (keras.initializers.RandomNormal(stddev=0.01), "randn(0.01)")]:
    h = make(init).fit(Xtr, ytr, epochs=10, validation_split=0.2, verbose=0)
    plt.plot(h.history["val_loss"], label=name)
plt.legend(); plt.xlabel("epoca"); plt.ylabel("val_loss")
plt.title("He gana con ReLU; randn(0.01) casi no arranca"); plt.show()

**Ej. 3 — Histogramas de activaciones.** Con un init adecuado (He), la varianza de las salidas se mantiene estable entre capas.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

inp = keras.Input((784,)); x = inp; outs = []
for u in (256, 128, 64):
    x = layers.Dense(u, activation="relu", kernel_initializer="he_normal")(x)
    outs.append(x)
probe = keras.Model(inp, outs)
acts = probe(tf.random.normal((512, 784)))
for i, a in enumerate(acts):
    plt.hist(a.numpy().ravel(), bins=50, alpha=0.5, label=f"capa {i} (var={a.numpy().var():.2f})")
plt.legend(); plt.title("He mantiene la varianza estable capa a capa"); plt.show()

**Ej. 4 — He + Tanh (combinación incorrecta).** He asume ReLU; con `tanh` el init correcto es Glorot.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def make(init):
    m = keras.Sequential([keras.Input((784,))] +
        [layers.Dense(128, activation="tanh", kernel_initializer=init) for _ in range(4)] +
        [layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for init, name in [("he_normal", "He (mal para tanh)"), ("glorot_uniform", "Glorot (correcto)")]:
    h = make(init).fit(Xtr, ytr, epochs=10, validation_split=0.2, verbose=0)
    plt.plot(h.history["val_loss"], label=name)
plt.legend(); plt.title("Glorot es el init correcto para tanh/sigmoid"); plt.show()
print("La eleccion del init depende de la activacion: He<->ReLU, Glorot<->tanh/sigmoid.")

**Ej. 5 — Reproducibilidad.** Con `tf.random.set_seed(42)` + `np.random.seed(42)` la inicialización es idéntica entre corridas.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def init_once():
    tf.random.set_seed(42); np.random.seed(42)
    m = keras.Sequential([keras.Input((784,)),
        layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax")])
    return m.layers[0].kernel.numpy()[0, :3]

a, b = init_once(), init_once()
print("run1:", a); print("run2:", b)
assert np.allclose(a, b)
print("Semilla fija -> inicializacion reproducible (clave para debuggear y comparar).")